In [ ]:
from kaggle_secrets import UserSecretsClient
import os
import glob
from datetime import datetime, timedelta

# 1. Lấy thông tin xác thực từ Kaggle Secrets
user_secrets = UserSecretsClient()
username = user_secrets.get_secret("EARTHDATA_USER")
password = user_secrets.get_secret("EARTHDATA_PASS")

# 2. Tạo file .netrc (Bắt buộc để NASA xác thực tự động)
netrc_path = os.path.expanduser("~/.netrc")
with open(netrc_path, "w") as f:
    f.write(f"machine urs.earthdata.nasa.gov login {username} password {password}\n")
os.chmod(netrc_path, 0o600)

# 3. Tạo file .urs_cookies để lưu phiên đăng nhập
cookie_path = os.path.expanduser("~/.urs_cookies")
if not os.path.exists(cookie_path):
    os.system(f"touch {cookie_path}")

# Tọa độ và biến số giữ nguyên theo chuỗi truy vấn OPeNDAP gốc của bạn
SUBSET_QUERY = "?Tb[0:1][1346:2432][7476:8851],time,lat[1346:2432],lon[7476:8851]"
TARGET_HOURS = ["00", "06", "12", "18"]

# Định nghĩa khoảng thời gian chạy từ 2020 đến 2025 (Chạy hết ngày 31/12/2025)
start_date = datetime(2020, 1, 1)
end_date = datetime(2026, 5, 30)

base_output_dir = "/kaggle/working/nasa_ir_data"
filtered_list_path = "/kaggle/working/filtered_nasa_links.txt"

# =====================================================================
# BẮT ĐẦU VÒNG LẶP QUA TỪNG NĂM ĐỂ SINH LINK VÀ TẢI DỮ LIỆU
# =====================================================================
current_year = start_date.year

while start_date <= end_date:
    year_str = str(start_date.year)
    output_dir = f"{base_output_dir}/year_{year_str}"
    
    if not os.path.exists(output_dir): 
        os.makedirs(output_dir)
        
    print(f"\n=================================================================")
    print(f"🌍 BẮT ĐẦU TẠO LINK VÀ TẢI DỮ LIỆU CHO NĂM {year_str} 🌍")
    print(f"=================================================================")
    
    links_to_download = []
    
    # Gom toàn bộ ngày thuộc năm hiện tại vào một danh sách để tải theo đợt năm
    while start_date <= end_date and start_date.year == current_year:
        year = start_date.strftime("%Y")
        month = start_date.strftime("%m")
        day = start_date.strftime("%d")
        
        # Tính toán Day of Year (001 - 366) dựa trên lịch trục thời gian
        day_of_year = f"{start_date.timetuple().tm_yday:03d}"
        
        # Sinh link cho 4 khung giờ cố định
        for hour in TARGET_HOURS:
            url = (
                f"https://disc2.gesdisc.eosdis.nasa.gov/opendap/MERGED_IR/GPM_MERGIR.1/"
                f"{year}/{day_of_year}/merg_{year}{month}{day}{hour}_4km-pixel.nc4.nc4{SUBSET_QUERY}"
            )
            links_to_download.append(url)
            
        start_date += timedelta(days=1)
        
    # Cập nhật mốc năm tiếp theo cho vòng lặp ngoài
    if start_date <= end_date:
        current_year = start_date.year

    # Ghi danh sách link của năm hiện tại ra file tạm để nạp vào wget
    count = len(links_to_download)
    print(f"--- Đã khởi tạo xong {count} link cho năm {year_str} ---")
    
    if count > 0:
        with open(filtered_list_path, 'w') as fout:
            for link in links_to_download:
                fout.write(link + "\n")
                
        print(f"--- Đang tiến hành tải dữ liệu vào: {output_dir} ---")
        # Gọi lệnh hệ thống wget để tải danh sách link
        # -nc: bỏ qua không tải lại nếu file đã tồn tại trên Kaggle
        os.system(f"wget --load-cookies ~/.urs_cookies --save-cookies ~/.urs_cookies --keep-session-cookies "
                  f"-nc -i {filtered_list_path} -P {output_dir}")
        
        # Tiến hành chuẩn hóa dọn dẹp đuôi ?Tb... loằng ngoằng sau khi tải xong cho năm này
        print(f"--- Đang chuẩn hóa tên file cho năm {year_str} ---")
        for f_path in glob.glob(f"{output_dir}/*?*"):
            new_name = f_path.split('?')[0]
            if not os.path.exists(new_name):
                os.rename(f_path, new_name)
                
        print(f"--- Hoàn thành xử lý thư mục: {output_dir} ---")

# Dọn dẹp file txt tạm sau khi hoàn tất toàn bộ tiến trình
if os.path.exists(filtered_list_path):
    os.remove(filtered_list_path)

print("\n🚀 --- HOÀN TẤT TOÀN BỘ TIẾN TRÌNH TẢI DỮ LIỆU NASA (2020 - 2025) --- 🚀")